# ATDL — Post-Task-4 End-to-End Ternary KD Research Notebook

**Purpose:** continue from the preserved Task-4 state and execute the strongest practical post-Task-4 research path under the project's existing research constitution.

This notebook is intentionally **append-only**. It does not overwrite Tasks 1–4, does not touch the frozen split, and does not evaluate the official test set. It is a research controller: it reuses the repository's existing model/quantizer/training/diagnostic implementations whenever available and provides a single place to run, compare, diagnose, tune, and finally lock the best method.

### Research path

`T3 no-KD → T4 vanilla KD → DKD → DIST → feature/attention KD → relational KD → QTRD/QFD-style quantization-aware transfer → STE/optimization probe → progressive/adaptive probe → targeted AutoML → confirmation → ablations → compression → final test only after explicit authorization`

### Important

- The notebook does **not** assume that every method will improve accuracy.
- Screening, confirmation, and headline claims are separated.
- Accuracy is not the only gate: KD, gradient, representation, quantization, and stability diagnostics are recorded.
- AutoML is used only after a mechanism family is selected; it is not allowed to silently search across unrelated causal factors.
- If an existing project script/API differs, the notebook detects and reuses it where possible; edit only the small configuration cell rather than rewriting the project.

In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, subprocess, shlex, re, math, statistics
from datetime import datetime
import numpy as np

# Change ONLY this path if needed.
PROJECT = Path("/home/vu-lab03-pc17/ATDL-1").resolve()
assert PROJECT.exists(), f"Project not found: {PROJECT}"

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print("PROJECT:", PROJECT)
print("Python:", sys.version)

## 0. Immutable research contract

The project constitution defines the hard controls: CIFAR-10, frozen 45k/5k split, frozen ResNet34 teacher, ResNet18 student, strict ternary deployed Conv/FC weights, QAT during training, frozen teacher, append-only artifacts, and a sealed test set. fileciteturn19file0

The current roadmap explicitly treats T3 and T4 as the controls and makes DKD/DIST/feature/relational and QFD/QTRD-style methods the main post-T4 research direction. fileciteturn20file2turn20file8

In [ ]:
# Read-only project inventory. This is deliberately targeted rather than a full codebase dump.
important = [
    "KD.md", "POST_TASK4_AUDIT.md", "RESEARCH_CONSTITUTION.md",
    "RESEARCH_LEDGER.md", "LITERATURE_TO_EXPERIMENT_MAP.md",
    "AUTOML_SEARCH_POLICY.md", "FAILURE_ANALYSIS.md",
    "RESEARCH_HYPOTHESES.md",
    "src", "scripts", "configs", "results", "experiments", "research_db"
]
for name in important:
    p = PROJECT / name
    print(f"{name:35} {'OK' if p.exists() else 'MISSING'}")

In [ ]:
# Existing artifacts — preserve them.
protected_patterns = [
    "results/**/*task3*", "results/**/*task4*",
    "experiments/**/*task3*", "experiments/**/*task4*",
    "plots/**/*task3*", "plots/**/*task4*"
]
for pat in protected_patterns:
    hits = list(PROJECT.glob(pat))
    if hits:
        print(pat, "=>", len(hits), "artifacts")

# 1. Experiment registry and safe runner

Every new run receives a unique name and an immutable configuration snapshot. The runner refuses to use the test loader for research runs and records stdout/stderr for reproducibility.

In [ ]:
RUN_ROOT = PROJECT / "experiments" / "post_task4_notebook"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
LEDGER_ROOT = PROJECT / "research_db" / "notebook_runs"
LEDGER_ROOT.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def config_hash(cfg):
    raw = json.dumps(cfg, sort_keys=True, default=str).encode()
    return hashlib.sha256(raw).hexdigest()[:16]

def run_cmd(cmd, run_name, env=None, timeout=None, allow_failure=False):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_id = f"{run_name}_{stamp}"
    outdir = RUN_ROOT / run_id
    outdir.mkdir(parents=True, exist_ok=False)
    rec = {
        "run_id": run_id, "run_name": run_name, "command": cmd,
        "started": datetime.now().isoformat(), "project": str(PROJECT)
    }
    (outdir/"command.json").write_text(json.dumps(rec, indent=2))
    print("$", cmd)
    p = subprocess.run(
        cmd, shell=True, cwd=PROJECT, env=({**os.environ, **(env or {})}),
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        timeout=timeout
    )
    (outdir/"stdout.log").write_text(p.stdout)
    rec["returncode"] = p.returncode
    rec["finished"] = datetime.now().isoformat()
    (outdir/"command.json").write_text(json.dumps(rec, indent=2))
    print(p.stdout[-6000:])
    if p.returncode != 0 and not allow_failure:
        raise RuntimeError(f"Command failed ({p.returncode}). See {outdir/'stdout.log'}")
    return p.returncode, outdir

def safe_name(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", x)

print("Notebook run root:", RUN_ROOT)

# 2. Preserve and audit Task 4 before continuing

Do not rerun Task 4 unless its final summary is genuinely missing. The current repository evidence says the fixed final reproduction is the pivotal T3-vs-T4 comparison and must remain immutable. fileciteturn19file3

In [ ]:
# Locate Task-4 summaries without modifying anything.
candidates = sorted(
    list(PROJECT.glob("results/**/*task4*summary*.json")) +
    list(PROJECT.glob("results/**/*t4*summary*.json"))
)
print("Task-4 summary candidates:")
for p in candidates[:30]:
    print(" ", p.relative_to(PROJECT))

# Load any obvious summary for display.
for p in candidates:
    try:
        obj = json.loads(p.read_text())
        if isinstance(obj, dict) and ("validation_mean" in obj or "validation_best" in obj):
            print(json.dumps(obj, indent=2)[:12000])
            break
    except Exception:
        pass

## Diagnostic gate

For every serious post-T4 experiment, collect:

- validation mean/std and best epoch
- CE and KD losses
- KD/CE contribution
- teacher/student agreement and confidence/entropy
- `||grad_CE||`, `||grad_KD||`, `||grad_total||`
- `cos(grad_CE, grad_KD)`
- layer-wise quantization error and sparsity
- representation similarity where available
- training stability and runtime

The research framework explicitly calls for a diagnostic gate rather than `experiment → accuracy → promote/reject`.

In [ ]:
# Generic numerical helpers for saved diagnostic records.
def cosine(a, b, eps=1e-12):
    a = np.asarray(a).ravel(); b = np.asarray(b).ravel()
    den = np.linalg.norm(a)*np.linalg.norm(b) + eps
    return float(np.dot(a,b)/den)

def summarize(values):
    x = np.asarray(values, dtype=float)
    return {
        "n": int(x.size),
        "mean": float(x.mean()) if x.size else None,
        "std": float(x.std(ddof=1)) if x.size > 1 else 0.0,
        "min": float(x.min()) if x.size else None,
        "max": float(x.max()) if x.size else None,
    }

def empirical_noise_reference():
    # Values supplied by the established project baseline; used only as context.
    return {
        "T3_mean": 0.9521, "T3_std": 0.0008,
        "T4_mean": 0.9521, "T4_std": 0.0007
    }
print(empirical_noise_reference())

# 3. Single-variable research ladder

The fastest rigorous route is not to exhaustively tune everything. Use:

1. **one-seed screening** for mechanisms,
2. **two-seed confirmation** for promising mechanisms,
3. **three-seed 200-epoch confirmation** for the final winner,
4. targeted AutoML only inside the winning mechanism family.

This follows the existing constitution's evidence gates. fileciteturn19file0

Recommended order:

**DKD → DIST → best feature/attention transfer → relational KD → QTRD/QFD-style attainable-target transfer → only then STE/progressive/adaptive optimization if diagnostics justify them.**

In [ ]:
# Configuration: adapt only the script path/CLI in this cell if the existing trainer uses a different interface.
TRAINER = PROJECT / "scripts" / "train_student_ternary.py"
print("Trainer:", TRAINER, "exists:", TRAINER.exists())

# Detect useful existing scripts rather than inventing replacements.
for pat in ["scripts/*.py", "src/**/*.py", "configs/**/*.yaml", "configs/**/*.yml"]:
    hits = list(PROJECT.glob(pat))
    print("\n", pat, len(hits))
    for p in hits[:40]:
        if any(k in p.name.lower() for k in ["kd","dist","dkd","ternary","quant","train","verify","diagnos","auto"]):
            print(" ", p.relative_to(PROJECT))

# 4. Task 5A — DKD screening

**Hypothesis:** decoupling target-class and non-target-class knowledge can provide a more useful signal than vanilla probability matching under the ternary bottleneck.

Do not stack DKD with feature losses yet.

Use the preserved T4 training recipe as the parent control and change only the KD mechanism plus its declared DKD weights.

Suggested screening:
- one seed
- 30–60 epochs
- α and β around the existing roadmap range `[0.5, 8]`
- temperature inherited from the best validated T4 setting first
- then only a small targeted α/β search if DKD itself shows promise.

The roadmap explicitly recommends DKD as a first probe. fileciteturn20file8

In [ ]:
# The repository's trainer is authoritative. This cell gives a conservative template.
# If the trainer exposes --kd-mode dkd, this can be used directly.
DKD_CMD = (
    "uv run python scripts/train_student_ternary.py "
    "--kd-mode dkd --epochs 50 --seeds 42 "
    "--run-name post4_dkd_screen "
    "--warm-start results/best_models/task4_best.pth"
)
print("Template command (DO NOT run blindly):")
print(DKD_CMD)

### DKD diagnostic interpretation

Promote DKD only if it shows a reproducible signal over T4/T3 **and** diagnostics indicate that the intended target/non-target decomposition is actually affecting the learning signal. A tiny validation gain alone is inconclusive.

# 5. Task 5B — DIST screening

**Hypothesis:** relational/logit-structure transfer may be more attainable than direct teacher probability matching for a discrete-weight student.

Run DIST independently from DKD. Do not combine them during the initial causal comparison.

In [ ]:
DIST_CMD = (
    "uv run python scripts/train_student_ternary.py "
    "--kd-mode dist --epochs 50 --seeds 42 "
    "--run-name post4_dist_screen "
    "--warm-start results/best_models/task4_best.pth"
)
print("Template command (adapt to the existing CLI):")
print(DIST_CMD)

# 6. Task 5C — feature/attention transfer

Only if DKD/DIST do not clearly win, or if their diagnostics show weak intermediate supervision, test normalized attention transfer.

Use stage-aligned features rather than arbitrary layer-by-layer matching. Avoid forcing raw feature MSE initially because ResNet34/ResNet18 have different depth and feature geometry.

Candidate:
- normalized attention maps
- stage-level alignment
- one small projection only if required
- feature loss weight γ as the only new numeric factor

In [ ]:
AT_CMD = (
    "uv run python scripts/train_student_ternary.py "
    "--kd-mode attention --epochs 50 --seeds 42 "
    "--run-name post4_attention_screen "
    "--warm-start results/best_models/task4_best.pth"
)
print("Template command (adapt to the existing CLI):")
print(AT_CMD)

# 7. Task 5D — relational KD

If direct feature/attention transfer is mismatched, test RKD/SP-style relational information.

Primary idea:

`teacher feature geometry → relation statistics → student relation geometry`

The student need not reproduce the teacher's exact FP32 feature values; it only needs to preserve useful structure.

In [ ]:
RKD_CMD = (
    "uv run python scripts/train_student_ternary.py "
    "--kd-mode rkd --epochs 50 --seeds 42 "
    "--run-name post4_rkd_screen "
    "--warm-start results/best_models/task4_best.pth"
)
print("Template command (adapt to the existing CLI):")
print(RKD_CMD)

# 8. Task 6 — Quantization-aware knowledge transfer (highest-value research branch)

This is the most important extension if time permits.

The core hypothesis is:

> The difficult part is not merely transferring teacher knowledge; it is transferring knowledge in a form the ternary student can represent.

The roadmap proposes a QFD/QTRD-style attainable-target mechanism: teacher features are transformed/quantized using the student's representational constraints before relational matching. This combines the attainable-target idea with relational transfer while keeping the mechanism explicit rather than becoming a kitchen-sink loss. fileciteturn20file8

Required ablation:

**raw teacher target vs student-quantized teacher target**

Everything else must remain fixed.

In [ ]:
QTRD_CMD = (
    "uv run python scripts/train_student_ternary.py "
    "--kd-mode qtrd --epochs 60 --seeds 42 "
    "--run-name post4_qtrd_screen "
    "--warm-start results/best_models/task4_best.pth"
)
print("Template command (adapt to the existing CLI):")
print(QTRD_CMD)

## QTRD diagnostic questions

For QTRD, do not merely ask whether accuracy improves.

Measure:

1. teacher feature → raw target discrepancy
2. teacher feature → quantized/attainable target discrepancy
3. student feature discrepancy
4. relation preservation
5. layer-wise quantization error
6. CE/KD gradient cosine
7. final accuracy

The strongest evidence would be:

`quantized target reduces unattainable mismatch → better-conditioned KD signal → better ternary student`

If that chain does not occur, do not claim it.

# 9. Mandatory gradient diagnostics

This notebook treats CE/KD gradient analysis as a first-class research instrument.

Required quantities:

`||grad_CE||`

`||grad_KD||`

`||grad_total||`

`||grad_KD|| / ||grad_CE||`

`cos(grad_CE, grad_KD)`

Negative cosine is evidence of conflict; near-zero cosine suggests weak alignment; positive cosine suggests compatible objectives.

These diagnostics should be computed on a fixed diagnostic minibatch or deterministic subset so comparisons are fair.

In [ ]:
# Standalone gradient-analysis utility.
# Integrate this function into the existing trainer/loss implementation rather than
# calling it independently during every training step.

def gradient_diagnostics(ce_grads, kd_grads):
    ce = np.concatenate([np.asarray(x).ravel() for x in ce_grads if x is not None])
    kd = np.concatenate([np.asarray(x).ravel() for x in kd_grads if x is not None])
    return {
        "grad_ce_norm": float(np.linalg.norm(ce)),
        "grad_kd_norm": float(np.linalg.norm(kd)),
        "kd_ce_norm_ratio": float(np.linalg.norm(kd)/(np.linalg.norm(ce)+1e-12)),
        "grad_ce_kd_cosine": cosine(ce, kd),
    }

# 10. Task 7 — STE/optimization probe

Only activate this branch if diagnostics suggest the loss is not the dominant problem.

Compare:

1. clipped STE — current control
2. identity STE — negative/control
3. ReSTE/annealed estimator if already implemented safely
4. optional gradient clipping/scaling

Change one mechanism at a time.

Do not simultaneously alter quantizer, KD loss, optimizer, and schedule.

In [ ]:
STE_CMD = (
    "uv run python scripts/train_student_ternary.py "
    "--kd-mode best "
    "--ste identity "
    "--epochs 50 --seeds 42 "
    "--run-name post4_ste_identity_screen"
)
print("Template only; adapt to actual CLI.")
print(STE_CMD)

# 11. Task 8 — progressive/curriculum ternarization

Only test this if training diagnostics indicate that immediate ternarization is an optimization bottleneck.

Candidate schedules:

- threshold annealing
- soft-to-hard ternarization
- short FP32/QAT warmup followed by immediate ternary deployment
- progressive quantization

**Control total training budget.** A curriculum must not secretly become "more training".

In [ ]:
PROGRESSIVE_CMD = (
    "uv run python scripts/train_student_ternary.py "
    "--kd-mode best --quant-schedule progressive "
    "--epochs 200 --seeds 42 "
    "--run-name post4_progressive_screen"
)
print("Template only; adapt to actual CLI.")
print(PROGRESSIVE_CMD)

# 12. Task 9 — adaptive knowledge transfer

Only after a stable mechanism is selected.

Potential adaptive signals:

- teacher/student disagreement
- confidence/entropy
- quantization difficulty
- layer-wise quantization error
- gradient alignment

Start with the simplest adaptive rule supported by diagnostics. Avoid a large dynamic λ/T search until there is evidence that fixed weighting is the problem.

In [ ]:
ADAPTIVE_CMD = (
    "uv run python scripts/train_student_ternary.py "
    "--kd-mode adaptive "
    "--epochs 60 --seeds 42 "
    "--run-name post4_adaptive_screen"
)
print("Template only; adapt to actual CLI.")
print(ADAPTIVE_CMD)

# 13. Targeted AutoML — only inside the selected mechanism

The existing AutoML policy recommends Optuna/TPE with Hyperband-style pruning, while keeping the search family controlled. fileciteturn20file2

For the selected mechanism, search only its relevant hyperparameters.

### Example: if QTRD wins

Search:

- temperature
- KD weight
- relation-loss weight
- feature-loss weight
- quantized-target choice
- possibly LR/WD in a **separate** optimization study

Do NOT jointly search:

`DKD + DIST + AT + RKD + STE + threshold + optimizer + schedule`

because then there is no clean mechanism attribution.

In [ ]:
# Optional Optuna bootstrap.
try:
    import optuna
    print("Optuna:", optuna.__version__)
except Exception as e:
    print("Optuna unavailable:", e)

AUTOML_DB = PROJECT / "experiments" / "automl"
AUTOML_DB.mkdir(parents=True, exist_ok=True)
print("AutoML root:", AUTOML_DB)

In [ ]:
# A generic objective skeleton. Connect objective() to the repository trainer.
# It deliberately returns validation accuracy only; all secondary diagnostics
# remain recorded separately so they do not create an arbitrary weighted score.

def automl_search_space(trial, family):
    if family == "dkd":
        return {
            "temperature": trial.suggest_categorical("temperature", [1,2,4,8,16]),
            "alpha": trial.suggest_float("alpha", 0.5, 8.0, log=True),
            "beta": trial.suggest_float("beta", 0.5, 8.0, log=True),
        }
    if family == "qtrd":
        return {
            "temperature": trial.suggest_categorical("temperature", [1,2,4,8,16]),
            "kd_weight": trial.suggest_float("kd_weight", 0.05, 1.0, log=True),
            "relation_weight": trial.suggest_float("relation_weight", 0.01, 1.0, log=True),
            "feature_weight": trial.suggest_float("feature_weight", 0.01, 1.0, log=True),
            "quant_target": trial.suggest_categorical("quant_target", ["raw","student_quantized"]),
        }
    raise ValueError(f"Unsupported family: {family}")

print("AutoML space defined for DKD and QTRD.")

# 14. Multi-seed confirmation

After screening, select at most 1–2 finalists.

Run:

- seeds 42, 43, 44
- 200 epochs
- identical teacher/split/student architecture
- identical ternary deployment constraint
- best validation checkpoint
- automatic ternary verification
- checkpoint reload verification

The headline result is the mean ± std, not the luckiest seed.

In [ ]:
# Confirmation command template.
CONFIRM_CMD = (
    "uv run python scripts/train_student_ternary.py "
    "--kd-mode qtrd "
    "--epochs 200 --seeds 42 43 44 "
    "--run-name final_candidate_confirmation"
)
print(CONFIRM_CMD)

# 15. Evidence-based promotion gate

Use this gate after each serious experiment.

### GREEN
- reproducible improvement over parent
- outside/above empirical seed noise where appropriate
- ternary verification passes
- no instability
- diagnostics support the proposed mechanism

### YELLOW
- gain within noise
- mixed diagnostics
- one seed wins but others do not
- mechanism evidence inconclusive

### RED
- degradation
- instability
- invalid ternary deployment
- test leakage
- teacher changed
- contaminated comparison
- implementation failure

A YELLOW result is not a failure; it means **do not claim improvement yet**.

In [ ]:
def promotion_gate(candidate_mean, parent_mean, candidate_std, parent_std,
                    mechanism_supported, ternary_ok=True, contaminated=False):
    if contaminated or not ternary_ok:
        return "RED"
    delta = candidate_mean - parent_mean
    # Conservative empirical screen: gains smaller than the combined 1-sigma
    # scale are not automatically promoted.
    noise = math.sqrt(candidate_std**2 + parent_std**2)
    if delta > noise and mechanism_supported:
        return "GREEN"
    if delta >= 0:
        return "YELLOW"
    return "RED"

# 16. Automated comparison table

Once experiment summaries exist, consolidate them into one table.

Required rows:

- T2 FP32 student
- T3 ternary no-KD
- T4 vanilla KD
- DKD
- DIST
- attention transfer
- RKD
- QTRD/QFD-style
- final selected method

Required columns:

validation mean/std, best validation, accuracy delta vs T3/T4, sparsity, quantization error, KD diagnostics, gradient cosine, runtime, decision.

In [ ]:
# Search JSON summaries and build a lightweight research table.
import pandas as pd

def collect_jsons():
    paths = list(PROJECT.glob("results/**/*.json")) + list(PROJECT.glob("experiments/**/*.json"))
    rows = []
    for p in paths:
        try:
            obj = json.loads(p.read_text())
        except Exception:
            continue
        if not isinstance(obj, dict):
            continue
        txt = (p.name + " " + str(obj.get("run_name",""))).lower()
        if any(k in txt for k in ["task3","task4","dkd","dist","qtrd","rkd","attention","adaptive","progressive"]):
            row = {"file": str(p.relative_to(PROJECT))}
            for k in ["task","run_name","validation_mean","validation_std","validation_best",
                      "sparsity_mean","best_epoch","test_evaluation","decision"]:
                if k in obj: row[k] = obj[k]
            rows.append(row)
    return pd.DataFrame(rows)

comparison_df = collect_jsons()
comparison_df.head(30)

# 17. Representation and prediction diagnostics

Use these when the selected method reaches confirmation.

### Prediction
- teacher/student agreement
- disagreement set
- confidence
- entropy
- target-class probability
- non-target mass
- logit margin

### Representation
- stage-wise cosine similarity
- feature norm ratio
- normalized attention similarity
- relation preservation
- optional CKA if computationally affordable

The objective is to determine whether an accuracy gain comes from genuinely improved knowledge transfer or merely optimization noise.

In [ ]:
# Generic metric implementations usable by the project diagnostic code.
def entropy_from_logits(logits, temperature=1.0):
    x = np.asarray(logits, dtype=np.float64) / temperature
    x = x - x.max(axis=-1, keepdims=True)
    p = np.exp(x); p /= p.sum(axis=-1, keepdims=True)
    return float((-p*np.log(np.clip(p,1e-12,1))).sum(axis=-1).mean())

def top1_agreement(teacher_logits, student_logits):
    return float((np.argmax(teacher_logits,1) == np.argmax(student_logits,1)).mean())

def mean_confidence(logits):
    x = np.asarray(logits, dtype=np.float64)
    x = x - x.max(axis=-1, keepdims=True)
    p = np.exp(x); p /= p.sum(axis=-1, keepdims=True)
    return float(p.max(axis=1).mean())

# 18. Quantization diagnostics

For every final candidate verify:

- every deployed Conv and FC is ternary
- deployed values are exactly `{−α, 0, +α}` per channel
- latent weights remain distinct from deployed weights
- sparsity
- α per layer/channel
- mean/relative quantization error
- no dead layer caused by thresholding

The project already has `verify_ternary.py`; use it as the authoritative verifier rather than duplicating its logic.

In [ ]:
VERIFY = PROJECT / "src" / "evaluation" / "verify_ternary.py"
if not VERIFY.exists():
    VERIFY = PROJECT / "scripts" / "verify_ternary.py"
print("Verifier:", VERIFY, VERIFY.exists())

def verify_checkpoint_template(checkpoint):
    if not VERIFY.exists():
        print("Verifier not found; locate the existing project verifier before finalization.")
        return
    cmd = f"uv run python {shlex.quote(str(VERIFY))} --checkpoint {shlex.quote(str(checkpoint))}"
    print(cmd)

# 19. Task 11 — efficient ablations

Do not repeat the entire universe of experiments.

For the final method, select the **few ablations that answer causal questions**:

1. no KD vs final KD
2. vanilla KD vs final KD
3. raw target vs quantized target (for QTRD/QFD)
4. one important temperature or KD-weight sweep
5. one quantizer/STE control only if it was implicated by diagnostics

This satisfies the research objective while avoiding an impractical combinatorial search.

In [ ]:
ABLATION_MATRIX = [
    {"id":"A0","parent":"T3","change":"none","purpose":"ternary no-KD control"},
    {"id":"A1","parent":"T4","change":"vanilla KD","purpose":"conventional KD control"},
    {"id":"A2","parent":"A1","change":"selected advanced KD","purpose":"mechanism comparison"},
    {"id":"A3","parent":"A2","change":"raw vs quantized target","purpose":"attainable-target ablation"},
    {"id":"A4","parent":"A2","change":"selected T or KD-weight sweep","purpose":"hyperparameter sensitivity"},
]
pd.DataFrame(ABLATION_MATRIX)

# 20. Task 12 — compression and deployment accounting

Report three different things separately:

1. **parameter count**
2. **theoretical weight-storage compression**
3. **actual measured runtime/memory**

For ternary weights, `log2(3) ≈ 1.585` bits/weight is the ideal information-theoretic weight representation before accounting for scale metadata and packing overhead.

Do **not** call latent FP32 storage a deployed model-size result.

Do **not** claim real wall-clock speedup merely from theoretical bit compression.

In [ ]:
def theoretical_ternary_bits(n_weights, n_channels=None, scale_bits=32):
    # Ideal code length plus a simple per-channel scale overhead estimate.
    bits = n_weights * math.log2(3)
    if n_channels:
        bits += n_channels * scale_bits
    return bits

def fp32_bits(n_weights):
    return n_weights * 32

def theoretical_compression_ratio(n_weights, n_channels=None):
    return fp32_bits(n_weights) / theoretical_ternary_bits(n_weights, n_channels)

print("Ideal ternary bits/weight:", math.log2(3))

# 21. Final statistical analysis

For the final winner:

- report all three seeds
- mean ± sample standard deviation
- best epoch per seed
- confidence interval if appropriate
- effect relative to T3 and T4
- do not promote based on one lucky run

If the difference is within observed seed noise, report it as **inconclusive**.

This is especially important because T3 and T4 are already around 95.21% and effectively tied.

In [ ]:
def ci95(values):
    x = np.asarray(values, dtype=float)
    if len(x) < 2:
        return (float(x.mean()), float(x.mean()))
    mean = x.mean()
    se = x.std(ddof=1)/math.sqrt(len(x))
    # t critical for df=2 ≈ 4.303; for 3 seeds.
    tcrit = 4.303 if len(x)==3 else 1.96
    return float(mean-tcrit*se), float(mean+tcrit*se)

print("Example CI for [0.952, 0.953, 0.954]:", ci95([.952,.953,.954]))

# 22. Test-set firewall

**Do not execute the final test evaluation from this notebook until the complete research configuration is frozen and explicitly authorized.**

Before final evaluation:

- freeze method
- freeze hyperparameters
- freeze checkpoint-selection rule
- freeze all ablations
- archive the research ledger
- verify no test loader was used in Tasks 5+
- verify final checkpoint is strictly ternary
- record the final configuration hash

Only then perform one authorized test evaluation.

In [ ]:
# Static firewall check: fail if research scripts explicitly import/call get_test_loader.
violations = []
for p in list((PROJECT/"scripts").glob("*.py")) + list((PROJECT/"src").rglob("*.py")):
    try:
        txt = p.read_text(errors="ignore")
    except Exception:
        continue
    if ("get_test_loader" in txt or "test_loader" in txt) and any(
        k in p.name.lower() for k in ["train","kd","quant","automl","research","task"]
    ):
        violations.append(str(p.relative_to(PROJECT)))

print("Potential research/test-loader references:")
for x in violations[:50]:
    print(" ", x)
print("NOTE: references require manual classification; do not auto-delete anything.")

# 23. Final research decision template

Fill this only after confirmation.

### Final method
`________________`

### Validation
`mean = ______`
`std = ______`
`best = ______`

### Relative to T3
`Δ = ______`

### Relative to T4
`Δ = ______`

### Mechanism evidence
- KD signal: ______
- gradient alignment: ______
- representation alignment: ______
- quantization error: ______
- stability: ______

### Ternary verification
`PASS / FAIL`

### Decision
`PROMOTE / INCONCLUSIVE / REJECT`

### Scientific conclusion

State **what the evidence supports**, not what you hoped would happen.

In [ ]:
# Persist a final decision record without overwriting any previous experiment.
final_record = {
    "notebook": "ATDL_post_task4_end_to_end_research",
    "created": datetime.now().isoformat(),
    "status": "template_only",
    "test_evaluation": "LOCKED",
    "immutable_controls": {
        "dataset": "CIFAR-10",
        "split": "frozen 45k/5k train/validation",
        "teacher": "frozen ResNet34 FP32",
        "student": "ResNet18",
        "deployed_weights": "ternary {-alpha,0,+alpha}",
    },
}
path = RUN_ROOT / "notebook_final_template.json"
path.write_text(json.dumps(final_record, indent=2))
print(path)

# 24. Minimal-time execution schedule

If compute/time is severely limited, execute **only this path**:

### Phase A — preserved controls
T3 is complete. Task 4 is **not** complete: preserve its completed seed 42 and complete fresh independent seeds 43 and 44 under the frozen protocol. Do not rerun or overwrite seed 42.

### Phase B — mechanism screen
1. DKD — 1 seed
2. DIST — 1 seed
3. QTRD/QFD-style attainable-target method — 1 seed

### Phase C — winner confirmation
Take the best mechanism and run:
- 2 seeds screening confirmation
- then 3 seeds × 200 epochs

### Phase D — targeted AutoML
Tune only the winning mechanism's:
- T
- KD weight
- mechanism-specific weights

### Phase E — final ablation
Do:
- T3 vs final
- T4 vs final
- raw vs quantized target if applicable
- one key hyperparameter sweep

### Phase F — final analysis
Diagnostics + compression + report.

This is the highest-value path when time is scarce. It preserves the project's research depth without spending most of the budget on low-probability branches.

# 25. Time-constrained execution controller — authoritative Task-4 completion

**Current state:** Task 4 is complete (3/3 seeds): 42=95.14%, 43=94.96%, 44=95.26%; matched mean 95.12% ± 0.15%. All three checkpoints, histories and diagnostics are protected historical anchors. The post-T4 stages now begin with evidence-gated DKD screening.

`RUN_MODE = TIME_CONSTRAINED` pauses broad AutoResearch/AutoML. The stages after Task 4 are documented and resumable but are not launched until the matched vanilla-KD control has its complete 3-seed result. The test set is locked.

In [ ]:
RUN_MODE = 'TIME_CONSTRAINED'  # Options: TIME_CONSTRAINED, FULL_AUTORESEARCH, DIAGNOSTIC_ONLY, FINAL_CONFIRMATION
RUN_TRAINING_FROM_NOTEBOOK = False  # Safety default: launch only after all gates pass.
TASK4_BASE_CONFIG = PROJECT / 'configs/kd/resnet18_ternary_vanillaKD_final_rerun1.yaml'
TASK4_REMAINING_CONFIG = PROJECT / 'configs/kd/resnet18_ternary_vanillaKD_remaining_rerun1.yaml'
TASK4_SEED42_HISTORY = PROJECT / 'experiments/task4/histories/task4_b3_final_t2_lam09_rerun1/seed42.json'
TASK4_SEED42_CHECKPOINT = PROJECT / 'experiments/task4/checkpoints/task4_b3_final_t2_lam09_rerun1/resnet18_ternary_kd_seed42.pth'
TASK4_VALIDATOR = PROJECT / 'scripts/validate_task4_continuation.py'
TASK4_TRAINER = PROJECT / 'scripts/train_student_vanilla_kd.py'
assert RUN_MODE == 'TIME_CONSTRAINED'
assert all(p.exists() for p in [TASK4_BASE_CONFIG, TASK4_REMAINING_CONFIG, TASK4_SEED42_HISTORY, TASK4_SEED42_CHECKPOINT, TASK4_VALIDATOR, TASK4_TRAINER])
print({'mode': RUN_MODE, 'completed_seeds': [42, 43, 44], 'task4_status': 'COMPLETE', 'test_evaluation': 'LOCKED'})

In [ ]:
# Integrity gate: protocol identity, completed-anchor presence and immutable checkpoint hash.
gate = subprocess.run([sys.executable, str(TASK4_VALIDATOR)], cwd=PROJECT, text=True, capture_output=True)
print(gate.stdout)
assert gate.returncode == 0, gate.stderr
seed42 = json.loads(TASK4_SEED42_HISTORY.read_text())
assert seed42['best_validation_accuracy'] == 0.9514 and seed42['best_epoch'] == 158
assert seed42['config']['test_evaluation'] == 'not_run'
print('PASS: preserved Task-4 seed 42 is a validation-only immutable anchor.')

In [ ]:
# Hard test firewall: research orchestration must neither load nor score the test set.
forbidden = ('get_test_loader(', 'CIFAR10Test', 'test_loader =')
research_sources = [TASK4_TRAINER, TASK4_VALIDATOR]
for source in research_sources:
    text = source.read_text()
    hits = [token for token in forbidden if token in text]
    assert not hits, f'Test-firewall violation in {source.name}: {hits}'
print('PASS: continuation trainer and validator contain no test-loader or test-metric access.')

In [ ]:
# Task 4 is complete. Do not rerun it: advance only through evidence-gated post-T4 stages.
TASK4_UNIFIED_SCORES = [0.9514, 0.9496, 0.9526]
print(f'Task 4 complete: {np.mean(TASK4_UNIFIED_SCORES):.4%} ± {np.std(TASK4_UNIFIED_SCORES, ddof=1):.4%} validation.')
print('Active next stage: DKD one-seed 50-epoch screen; promote only after saved diagnostics are reviewed.')

In [ ]:
# Durable hand-off record: run after each major stage; never changes prior experiment artifacts.
def save_research_state_checkpoint():
    state = {'timestamp': datetime.now().isoformat(), 'run_mode': RUN_MODE,
             'completed_tasks': ['Task 1', 'Task 2', 'Task 3'],
             'task4_seed_completion': {'complete': [42, 43, 44], 'remaining': []},
             'task4_validation': {'mean': 0.9512, 'std': 0.00151, 'per_seed': {42: 0.9514, 43: 0.9496, 44: 0.9526}},
             'test_evaluation': 'LOCKED_NOT_RUN',
             'automl_status': 'PAUSED_NOT_ABANDONED',
             'research_ledger': str(PROJECT / 'research_db/experiments.jsonl'),
             'next_experiment': 'Review the live DKD screen; promote only if its validation and diagnostics justify further compute.'}
    path = PROJECT / 'research_state_checkpoint.json'
    if path.exists():
        path = PROJECT / f'research_state_checkpoint_{datetime.now():%Y%m%d_%H%M%S}.json'
    path.write_text(json.dumps(state, indent=2) + '\n')
    return path
print('Call save_research_state_checkpoint() after the current stage completes.')

## Expected scientific endpoint

Do not define success as "must beat 95.21%".

A successful research outcome can be any of:

1. a reproducible accuracy improvement;
2. evidence that a specialized KD mechanism is better suited to the ternary bottleneck;
3. evidence that vanilla KD does not help and why;
4. evidence that quantization-aware/attainable targets improve the KD signal;
5. evidence that the limiting factor is optimization/STE rather than KD;
6. a strong negative result that rules out plausible mechanisms.

The strongest final story is:

**teacher → knowledge representation → ternary attainability → diagnostics → controlled improvement/falsification.**

This is more scientifically valuable than an uncontrolled hyperparameter sweep.